# 04 - Entrenamiento y Optimizacion

**TP2 - Modulo 2 - Clasificacion `smoking`**

Aca entrenamos el modelo. Separamos los datos en train y validacion, y buscamos los mejores hiperparametros para XGBoost probando 30 combinaciones con validacion cruzada. La metrica que optimizamos es el F1 de la clase 1.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, make_scorer
from xgboost import XGBClassifier

import utils
from utils import SEED, DATA_PROCESSED, MODELS, MODELS_XGB

df = pd.read_csv(DATA_PROCESSED / 'train_clean.csv')
X, y = utils.features_target(df)
print('X:', X.shape, '| y balance:', dict(y.value_counts()))

X: (50000, 32) | y balance: {0: np.int64(31671), 1: np.int64(18329)}


## Separacion train / validacion

Reservamos el 20% como validacion. Usamos `stratify=y` para que la proporcion de fumadores sea la misma en los dos grupos. Guardamos el split para que el notebook 05 use exactamente los mismos datos.

In [2]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

print('train:', X_train.shape, '| val:', X_val.shape)

# Persistimos el hold-out para la notebook de validacion.
X_val.to_csv(DATA_PROCESSED / 'X_val.csv', index=False)
y_val.to_csv(DATA_PROCESSED / 'y_val.csv', index=False)

train: (40000, 32) | val: (10000, 32)


## Configuracion

Optimizamos F1 de la clase 1 con validacion cruzada estratificada de 5 folds. `scale_pos_weight` le dice al modelo cuanto mas peso darle a la clase minoritaria (fumadores).

In [3]:
f1_pos = make_scorer(f1_score, pos_label=1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
spw = n_neg / n_pos
print('scale_pos_weight =', round(spw, 3))

prep_arbol = joblib.load(MODELS / 'preprocessor_arbol_unfitted.joblib')

scale_pos_weight = 1.728


## XGBoost

In [4]:
pipe_xgb = Pipeline([
    ('prep', prep_arbol),
    ('clf', XGBClassifier(
        objective='binary:logistic', eval_metric='logloss',
        tree_method='hist', random_state=SEED, n_jobs=-1)),
])
grid_xgb = {
    'clf__n_estimators': [200, 400, 600],
    'clf__max_depth': [4, 5, 6, 8],
    'clf__learning_rate': [0.03, 0.05, 0.1, 0.2],
    'clf__subsample': [0.7, 0.85, 1.0],
    'clf__colsample_bytree': [0.7, 0.85, 1.0],
    'clf__min_child_weight': [1, 3, 5],
    'clf__reg_lambda': [1.0, 3.0, 5.0],
    'clf__scale_pos_weight': [1.0, spw],
}
search_xgb = RandomizedSearchCV(pipe_xgb, grid_xgb, n_iter=30, scoring=f1_pos,
                                cv=cv, random_state=SEED, n_jobs=-1, verbose=1)
search_xgb.fit(X_train, y_train)
print('XGB best F1 (cv):', round(search_xgb.best_score_, 4), '| params:', search_xgb.best_params_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


XGB best F1 (cv): 0.7301 | params: {'clf__subsample': 0.85, 'clf__scale_pos_weight': 1.7279547159517152, 'clf__reg_lambda': 3.0, 'clf__n_estimators': 600, 'clf__min_child_weight': 1, 'clf__max_depth': 8, 'clf__learning_rate': 0.05, 'clf__colsample_bytree': 0.85}


## Guardado

In [5]:
MODELS_XGB.mkdir(parents=True, exist_ok=True)

joblib.dump(search_xgb.best_estimator_, MODELS_XGB / 'xgb_pipeline.joblib')

cv_results = pd.DataFrame({
    'modelo': ['XGBoost'],
    'f1_clase1_cv': [search_xgb.best_score_],
})
cv_results.to_csv(MODELS / 'cv_results.csv', index=False)
cv_results

,modelo,f1_clase1_cv
0,XGBoost,0.730097


## Resultado

XGBoost con los mejores hiperparametros encontrados dio un F1 de clase 1 de **0.7301** en validacion cruzada. El ajuste fino del threshold y la evaluacion final van en el notebook 05.